# Build model — OLD aggregator (DEEP tree)

Same data/flow as `Build_Model_Old.ipynb`, but with a **deeper, less-regularized XGBoost** — the OLD-aggregator
counterpart to `Build_Model_New_Deep.ipynb`.

**Why:** the lightweight tree (depth 3, `colsample_bytree=0.05`, `min_child_weight=250`) is so throttled it
leans on the unchanged delinquency *count* features and never has to use the `percent_of_DQ` features the
aggregator fix actually moves. A deeper, feature-hungrier tree gives the fix a chance to show up if it can.
Both NEW and OLD use the **identical** deep config so any gap is still attributable to the aggregator alone.

- **trade** = `Features_To_Use.json` subset (the percent-of-DQ features), **app** = all cols, **target** = `final_DQ60_m24`
- train Q2'19, evaluate test rows via `data_split`
- LevelSelection -> FillNA -> XGBoost (**depth 8, n_estimators 800, colsample 0.3, min_child_weight 50**)

Output -> `payment_processing_research_data/models/model_old_deep/` (originals untouched). Model-engine kernel.

In [1]:
import os, json, importlib
import model_configs
importlib.reload(model_configs)
from model_engine.model_builder.build_model import build_model

VARIANT = 'old'
out_dir = os.path.join(model_configs.MODELS_DIR, f'model_{VARIANT}_deep')
os.makedirs(out_dir, exist_ok=True)
print('variant:', VARIANT, '| DEEP | output ->', out_dir)

variant: old | DEEP | output -> /home/jag/payment-processor-research/payment_processing_research_data/models/model_old_deep


In [2]:
# Deeper, less-regularized pipeline (override the lightweight one in model_configs).
# Same LevelSelection -> FillNA front end; only the XGBoost knobs change.
# MUST match Build_Model_New_Deep.ipynb exactly so the only difference is new vs old data.
DEEP_PIPELINE_FACTORY = {
    'transformers': [
        {'zaml_class': 'LevelSelection',
         'params': {'thresh': 0.01, 'change_to': 'Other', 'encoding': 'onehot'}},
        {'zaml_class': 'FillNA',
         'params': {'replace_by': -1, 'add_flags': False}},
    ],
    'model': {
        'zaml_class': 'XGBoostModel',
        'params': {
            'learning_rate':     0.03,
            'n_estimators':      800,
            'max_depth':         8,      # was 3
            'backend_subsample': 0.7,
            'scale_pos_weight':  2.5,
            'colsample_bytree':  0.30,   # was 0.05 -> trees can actually see the percent features
            'min_child_weight':  50,     # was 250 -> finer splits allowed
        },
    },
}

asset = model_configs.build_asset(VARIANT)
asset['config']['pipeline_factory'] = DEEP_PIPELINE_FACTORY
print('xgb params:', asset['config']['pipeline_factory']['model']['params'])

xgb params: {'learning_rate': 0.03, 'n_estimators': 800, 'max_depth': 8, 'backend_subsample': 0.7, 'scale_pos_weight': 2.5, 'colsample_bytree': 0.3, 'min_child_weight': 50}


In [3]:
# sanity: every table has files before we train
for tbl in ['app', 'trade', 'target']:
    n = len(asset['data'][tbl]['data'])
    print(f'{tbl:7}: {n} files')
    assert n > 0, f'no {tbl} files -- run the fix_processing/Sample_* notebooks first'
print('trade keep_features:', 'ALL' if asset['data']['trade']['io_params']['keep_features'] is None
      else len(asset['data']['trade']['io_params']['keep_features']))
print('data_split:', asset['config']['data_split'])

json.dump(asset, open(os.path.join(out_dir, 'asset.json'), 'w'), indent=2)
print('wrote', os.path.join(out_dir, 'asset.json'))

app    : 6 files
trade  : 6 files
target : 6 files
trade keep_features: 1143
data_split: {'train': {'start_date': '2019-04-01', 'end_date': '2019-07-01'}, 'test': {'start_date': '2019-07-01', 'end_date': '2020-04-01'}}
wrote /home/jag/payment-processor-research/payment_processing_research_data/models/model_old_deep/asset.json


In [4]:
import logging, traceback

log_path = os.path.join(out_dir, f'build_model_{VARIANT}_deep.log')
fh = logging.FileHandler(log_path, mode='w')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
root = logging.getLogger(); root.addHandler(fh); root.setLevel(logging.INFO)

try:
    build_model(asset, out_dir)
    root.info('BUILD SUCCEEDED -> %s', out_dir)
    print('\nDONE -> model artifacts in', out_dir)
    print(sorted(os.listdir(out_dir)))
except Exception as e:
    root.error('BUILD FAILED: %s\n%s', e, traceback.format_exc())
    print('BUILD FAILED -- see log:', log_path)
    raise
finally:
    root.removeHandler(fh); fh.close()
    print('log saved ->', log_path)

parsing model-builder asset


Configuring model builder


INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_data>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <monotonic_constraints_list>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <add_default_monotonic_constraints>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <add_default_monotonic_constraints>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <fe_version>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <data_split>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <train_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <train_sample_weight>, thus it will be omitt

building model


INFO:zaml.artifact_engine.logger:Finished <versions>, total time spent: 0:00:04.602138
INFO:zaml.artifact_engine.logger:Executing MonotonicConstraintsListParser <parsed_monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Finished <parsed_monotonic_constraints_list>, total time spent: 0:00:01.003618
INFO:zaml.artifact_engine.logger:Executing SplitterArtifact <splitter>...
INFO:zaml.artifact_engine.logger:Finished <splitter>, total time spent: 0:00:00.000502
INFO:zaml.artifact_engine.logger:Executing DataArtifact <data>...
INFO:zaml.artifact_engine.logger:Finished <data>, total time spent: 0:00:00.000445
INFO:zaml.artifact_engine.logger:Executing ExclusionListParser <parsed_exclusion_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <parsed_exclusion_list>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Finished <parsed_exclusion_list>, total time spent: 0:00:00.000468
INFO:zaml.artifact_engine.logger:Executing Bi

...

-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 1143
Time spent: 571.697s
-------------------------
Name: app
Transformer type: None
Number of features: 52
Time spent: 0.000s
-------------------------
Name: trade
Transformer type: None
Number of features: 1143
Time spent: 0.000s
-------------------------
Name: app FE
Transformer type: OneToOneEngine
Number of features: 0
Time spent: 1.259s
-------------------------
Name: trade FE
Transformer type: EndtoEndFeatureEngine
Number of features: 1143
Time spent: 2.781s
-------------------------
Name: Merge data
Transformer type: Concat
Number of features: 1143
Time spent: 12.344s
-------------------------
Name: LevelSelection
Transformer type: LevelSelection
Number of features: 1143
Time spent: 3.891s


INFO:zaml.artifact_engine.logger:Finished <test_fe_data>, total time spent: 0:10:01.669334
INFO:zaml.artifact_engine.logger:Executing StaticAssetArtifact <static_asset>...
INFO:zaml.artifact_engine.logger:Finished <static_asset>, total time spent: 0:00:00.190515
INFO:zaml.artifact_engine.logger:Executing TrainHistoryArtifact <train_history>...
INFO:zaml.artifact_engine.logger:Finished <train_history>, total time spent: 0:00:00.000324
INFO:zaml.artifact_engine.logger:Executing BestModelParamsArtifact <best_model_params>...
INFO:zaml.artifact_engine.logger:Finished <best_model_params>, total time spent: 0:00:00.000270
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <train_scores>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 1143
Time spent: 581.340s


INFO:zaml.artifact_engine.logger:Finished <train_scores>, total time spent: 0:00:08.396322
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <train_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <train_submodel_scores>, total time spent: 0:00:00.000326
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <test_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_scores>, total time spent: 0:00:08.412710
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <test_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_submodel_scores>, total time spent: 0:00:00.000355
INFO:zaml.artifact_engine.logger:Executing CalibrationObjectArtifact <calibration_object>...
INFO:zaml.artifact_engine.logger:Finished <calibration_object>, total time spent: 0:00:11.282346
INFO:zaml.artifact_engine.logger:Executing FeatureDefinition <feature_definition>...
INFO:zaml.artifact_engine.logger:Finished <feature_definition>, total time spent: 0:0


DONE -> model artifacts in /home/jag/payment-processor-research/payment_processing_research_data/models/model_old_deep
['artifact_manifest.json', 'asset.json', 'best_model_params.json', 'build_model_old_deep.log', 'calibration_object.obj', 'feature_definition.parquet', 'feature_importance.parquet', 'fit_time_info.json', 'keep_features.json', 'key_factors_mapping.json', 'model.obj', 'model_strategy.json', 'mrm_pipeline.obj', 'parsed_monotonic_constraints_list.json', 'pipeline.obj', 'score_recalibration_mapping.json', 'splitter.obj', 'static_asset.json', 'test_app.parquet', 'test_auc.json', 'test_data_summary.json', 'test_fe_data.parquet', 'test_ks.json', 'test_scores.parquet', 'test_target.parquet', 'test_zest_scores.parquet', 'top_features.parquet', 'train_app.parquet', 'train_auc.json', 'train_data_summary.json', 'train_fe_data.parquet', 'train_history.json', 'train_ks.json', 'train_scores.parquet', 'train_target.parquet', 'train_zest_scores.parquet', 'value_based_key_factor_mapping.